# T Complexity Of Comparison Gates

<p style="text-align: center;"><a href="mailto:noureldinyosri@gmail.com">Noureldin Yosri</a></p>
<p style="text-align: center;">May 2023</p>

Native adaptation of the frozen Qualtran v0.7.0 tutorial at bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2023 Google LLC, Apache-2.0.

## Abstract
Quantum comparison gates can be split into two categories: quantum-classical when comparing a quantum number to a classical number and quantum-quantum when comparing two quantum numbers (i.e. quantum registers).

Comparison gates are an important building block for quantum algorithms, yet they are always relegated to the appendices usually with figures showing how they work only on a small number of qubits. This leads to cases where researchers potentially waste time reinventing already existing methods.
This notebook intends to serve as a documentary of the current state of the art for comparison gates while explaining how they work both algorithmically and programmatically.

**Note:** Most of the ideas explained here are built into [qualtran](https://github.com/quantumlib/Qualtran/tree/main/qualtran).

## Introduction
The current optimal implementation of a reversible oracle for comparing two quantum numbers of $n$ qubits each is $4n + \mathcal{O}(1)$ T operations. This is done by reducing the problem to subtraction which takes $4n + \mathcal{O}(1)$ (see. [Gidney, 2018](https://arxiv.org/abs/1709.06648)). 

This way while optimal in terms of T count has linear depth. The question of whether we can trade some T operations in order to reduce the depth of the circuit is interesting and turns out to have multiple answers.
In this notebook, we consider the problem of doing a comparison at a depth logarithmic in the number of qubits $\mathcal{O}(\log{n})$. The current best way to do this is given in the supplementary materials of [Berry et al., 2018](https://doi.org/10.1038/s41534-018-0071-5) and leverages the divide and conquer technique. 

When this decomposition is applied to the case where one input is a list of qubits and the other is a classical number (i.e. a string of bits) this decomposition gives a $6n + \mathcal{O}(1)$ T complexity.

In what follows I will explain the current T optimal ways for comparison in all cases. Our focus on T count is because T count is the bottleneck when running quantum circuits in a fault-tolerant way using surface code (see. [Litinski et al., 2018](https://doi.org/10.48550/arXiv.1808.02892))

$$\renewcommand{\ket}[1]{|#1\rangle}$$

## Equality as a special case
Before we proceed to the comparison oracle we take a look at the equality oracle actually as a special case with a T complexity of only $4n + \mathcal{O}(1)$, as it can be implemented as a qubit-wise And operation. This qubit-wise `And` operation itself can be done using only $4n + \mathcal{O}(1)$ as per [Babbush et al., 2018](https://doi.org/10.1103%2Fphysrevx.8.041015) and [Craig Gidney](https://algassert.com/post/1903).

### Quantum-Classical Case
In the quantum-classical case, all we need to do is to compute qubit/bit-wise equality
$$(q_0 = b_0) \wedge \cdots \wedge (q_{n-1} = b_{n-1})$$

In `cirq-ft` this can be accomplished using the `And` gate as `And(cv=bits)` where `bits` are the bits of the classical number.

### Quantum-Quantum Case
In the quantum-quantum case the equality is still bitwise $$(q_0 = p_0) \wedge \cdots \wedge (q_{n-1} = p_{n-1})$$

This is the same as the and $$(q_0 \oplus p_0 = 0) \wedge \cdots \wedge (q_{n-1}\oplus p_{n-1} = 0)$$

Where $\oplus$ is the binary xor operation. This allows us to use `qualtran`'s `And` on the result of the xor.

The finite equality cost for the examples is 4(n−1) T, using n−1 logical And computations.

In [ ]:
from examples.algorithms.comparison_cost_tutorial import (
    check_comparison, classical_mixer, comparison_tree, constant_equality,
    linear_constant_comparison, one_bit_comparison, register_equality,
    show_counts, simple_mixer, two_bit_classical_comparison, two_bit_comparison,
)

In [ ]:
equality_oracle_quantum_classical = constant_equality

As an example, we construct the equality gate for checking if a 3-qubit register is equal to 5. First, we print the decomposition of the gate followed by the result of running the gate on each of the 8 possibilities individually and finally the result of running the gate on the uniform superposition of all 8 possibilities.

In [ ]:
equality_circuit = equality_oracle_quantum_classical()
equality_circuit.instructions

In [ ]:
show_counts(equality_circuit, source_t=4 * (3 - 1))

In [ ]:
def check_basis(circuit, inputs, outputs, predicate):
    check_comparison(circuit, inputs, outputs, predicate, uniform=False)

In [ ]:
check_basis(equality_circuit, 3, 1, lambda v: int(v == 5))

As this is a quantum circuit, it's important to check that it works with superpositions as well. This is why we will check the uniform superposition.

In [ ]:
def check_uniform(circuit, inputs, outputs, predicate):
    check_comparison(circuit, inputs, outputs, predicate, uniform=True)

In [ ]:
check_uniform(equality_circuit, 3, 1, lambda v: int(v == 5))

And for the quantum-quantum case, we have.

In [ ]:
equality_oracle_quantum_quantum = register_equality

As we did before we construct the gate to compare two 2-qubit numbers. First, we print the decomposition of the gate followed by the result of running the gate on each of the 16 possibilities individually and finally the result of running the gate on the uniform superposition of all 16 possibilities.

In [ ]:
quantum_quantum_equality = equality_oracle_quantum_quantum()
quantum_quantum_equality.instructions

In [ ]:
show_counts(quantum_quantum_equality, source_t=4 * (2 - 1))

In [ ]:
check_basis(quantum_quantum_equality, 4, 1, lambda v: int(v >> 2 == v & 3))

In [ ]:
check_uniform(quantum_quantum_equality, 4, 1, lambda v: int(v >> 2 == v & 3))

Notice that the ancilla qubit are always clean at the end of execution and that the input qubits are not affected

## Definition of the Comparison Operator

Before we proceed let's formally define the the operator we want to implement. The operator is the less-than-operator. We chose to focus on this case since the $\leq$ and greater than cases differ by only Clifford operations. 

There are two versions of this operator. The first is the quantum-quantum case where we compare two-qubit register $\ket{p}$ and $\ket{q}$ and store the result in qubit $z$.
$$\hat{O}\ket{p}\ket{q}\ket{z} = \ket{p}\ket{q}\ket{z \oplus (p < q)}$$

The second is the quantum-classical case, where we compare a quantum register $\ket{p}$ to classical number $v$ (i.e. string of bits)
$$\hat{O}_v\ket{p}\ket{z} = \ket{p}\ket{z \oplus (p < v)}$$


## The Quantum-Quantum Comparator $8n + \mathcal{O}(1)$ T gates
For this case, the optimal decomposition in T count with logarithmic depth was proposed by [Berry et al., 2018](https://doi.org/10.1038/s41534-018-0071-5) and has $8n + \mathcal{O}(1)$ T complexity. This decomposition leverages the divide and conquer technique and builds a binary tree of depth $\log_2{(n)}$ whose leaves are the qubits of the numbers with intermediate values stored in $\mathcal{O}(n)$ ancillas. Each non-leaf node uses two CSWAP operations and the value of the comparison is computed from the root node using one Toffoli. 

Before we get into the implementation detail of this gate we will discuss two special cases $n=1$ and $n=2$ because as we will see later the general case is built out of these cases.

### $n = 1$ Case

Given 2 qubits, we would like a way to represent the three cases of equality, less than, greater than in a reversible way. This can be done using two extra qubits `less_than` and `greater_than` which will hold the less than and greater than while equality will be held in the second operand (qubit). In other words implementing the operator
$$\hat{O} \ket{p}\ket{q}\ket{0}\ket{0} \rightarrow \ket{p}\ket{p=q} \ket{p < q} \ket{p > q}$$

Note that this is different from our original goal to compute $$\hat{O}\ket{p}\ket{q}\ket{z} = \ket{p}\ket{q}\ket{z \oplus (p < q)}$$However our goal can be computed from this operator using a `CNOT` with its control being the first ancilla and its target being qubit $z$.

In [ ]:
def relation(a, b):
    return 4 * (a < b) + 2 * (a == b) + (a > b)

In [ ]:
single_qubit_compare_circuit = one_bit_comparison()
single_qubit_compare_circuit.instructions

In [ ]:
check_basis(single_qubit_compare_circuit, 2, 3, lambda v: relation(v >> 1, v & 1))

In [ ]:
check_uniform(single_qubit_compare_circuit, 2, 3, lambda v: relation(v >> 1, v & 1))

A slightly different circuit is given in the paper which we implemented in `qualtran` as `SingleQubitCompare` and has a cost of only one CCNOT (i.e. Toffoli = 4T).

`SingleQubitCompare` is implemented using measurement-based uncomputation so that its adjoint has a T count of zero.

The illustrative one-bit circuit above retains its two coherent CCX leaves. The two-bit illustration also retains coherent CSWAP leaves. Their physical T cost depends on the chosen lowering. The measured tree below expands the original optimized mixers.

### $n=2$ Case
In this case the two qubit numbers $P$ and $Q$ can be written as
$$
P = 2*p_1 + p_0\\
Q = 2*q_1 + q_0
$$

We will do this in two steps. First we update qubits such that $\mathrm{sign}(P - Q) = \mathrm{sign}(p_0^f - q_0^f)$ where $p_0^f$ and $q_0^f$ are the final values of $p_0$ and $q_0$ respectively. This way the comparison result between the two numbers can be extracted using `SingleQubitCompare` applied on $p_0^f$ and $q_0^f$.

Notice that if $p_1 = q_1$ then we don't need to do anything since $p_0$ and $q_0$ are already in the required state, but when $p_1 \neq q_1$ then all we need to do is to swap the qubits of each number since the result of comparing $P$ and $Q$ is the same as comparing $p_1$ and $q_1$. 

In [ ]:
simple_mixer()

In  [Berry et al., 2018](https://doi.org/10.1038/s41534-018-0071-5) the circuit that does the first step is called `COMPARE2` (FIG. 1 in the supplementary materials) and we implemented it in `qualtran` as `BiQubitsMixer`. This circuit is different from the simple circuit above. Yet both are equivalent as we will see below with this circuit having a smaller number of Clifford operations.

Each of the CSWAP operations has a T count of 4. We again utilized measurement-based uncomputation so that the adjoint of `BiQubitsMixer` has zero T count.

In [ ]:
compare_two_circuit = two_bit_comparison()
compare_two_circuit.instructions

In [ ]:
check_basis(compare_two_circuit, 4, 3, lambda v: relation(v >> 2, v & 3))

In [ ]:
check_uniform(compare_two_circuit, 4, 3, lambda v: relation(v >> 2, v & 3))

Now notice that for $n > 2$ we can group the qubits in $\frac{n}{2}$ pairs and use our COMPARE2 circuit to compare them, in other words:
$$
   \textit{Compare2}((P_0, P_1), (Q_0, Q_1)), \cdots, \textit{Compare2}((P_{n-2}, P_{n-1}), (Q_{n-2}, Q_{n-1}))
$$

The comparison result of the original sequences $P$ and $Q$ of length $n$ is preserved in this $\frac{n}{2}$ sequence.

Doing this $\log_2{n}$ times gives a circuit that has a tree structure (FIG. 2 of the supplementary materials of [Berry et al., 2018](https://doi.org/10.1038/s41534-018-0071-5)) of depth $\log_2{n}$ where each layer/depth contains independent operations. So the circuit has logarithmic depth.

Note that as in the case of 2 qubits, we still need a final `SingleQubitCompare` operation to extract the result.

In [ ]:
quantum_compare = comparison_tree(3)
quantum_compare.instructions

In [ ]:
show_counts(quantum_compare, source_t=8 * (3 - 1) + 4)

In [ ]:
run_check = False
if run_check:
    check_basis(quantum_compare, 6, 3, lambda v: relation(v >> 3, v & 7))

In [ ]:
check_uniform(quantum_compare, 6, 3, lambda v: relation(v >> 3, v & 7))

### T Complexity Analysis

The tree from above has $n-1$ nodes. Each of these nodes does a `BiQubitsMixer` and its adjoint gives a T count of $8 + 0 = 8$ per node. The final step is a `SingleQubitCompare` which has a T count of $4 + 0 = 4$. This gives a total of $8(n-1) + 4 = 8n-4$ T operations.

Note that the adjoints of `SingleQubitCompare` and `BiQubitsMixer` have zero T count as discussed above.

## $\mathcal{O}(6n)$ Quantum-Classical Comparator with logarithmic depth

We can use the tree structure we have for this case to achieve the desired depth. However, we need a new version of `BiQubitsMixer` that expects just one quantum number with the other being classical. This new version will have the advantage of processing a classical number so it can tweak its structure to reduce its T count.

Notice that our original `BiQubitsMixer` has two `CSWAP` operations with one of them swapping the two qubits of $Q$ but in our case, $Q$ is just a classical number and the CSWAP operation degenerates into either identity when the two bits of the numbers are equal (e.g. $Q \in \{0, 3\}$) or a CNOT when they are different (e.g. $Q \in \{1, 2\}$). This reduces the T count by half going from 8 to 4.

In [ ]:
quantum_classical_mixer = classical_mixer

In [ ]:
mixer, _ = quantum_classical_mixer(2)
mixer

In [ ]:
circuit = two_bit_classical_comparison(2)
circuit.instructions

In [ ]:
show_counts(circuit, source_t=4 + 4)

In [ ]:
check_basis(circuit, 2, 3, lambda v: relation(v, 2))

In [ ]:
check_uniform(circuit, 2, 3, lambda v: relation(v, 2))

#### T Complexity Analysis

Now that we have our `quantum_classical_mixer` we can use it in our tree. However, we can't use it except at the last layer of the tree (leaf nodes), since that's the only layer that sees the classical number. This means that only this layer can benefit from the classical information. The number of nodes in this layer is $\frac{n-1}{2}$ since this is a binary tree with $n-1$ nodes.

Putting all of it together we get 4T for the $\frac{n-1}{2}$ leaf nodes, 8T for the $\frac{n-1}{2}$ internal nodes, 4T for the final `SingleQubitCompare` or

$$4 \frac{n-1}{2} + 8\frac{n-1}{2} + 4 = 6n + \mathcal{O}(1)$$

Where the constant ($\approx -2$) depends only on the parity of $n$.

The two-bit mixed example actually reports 8 T. This cell preserves the original asymptotic discussion with the example cost recorded separately.

## Bonus: The quantum-classical comparator with $4n + \mathcal{O}(1)$ T gates and linear depth

### Inspiration

[Berry et al., 2019](https://doi.org/10.48550/arXiv.1902.02134) reduced the problem to subtraction, we however take a different approach. Note that comparing two numbers of equal size is the same finding which of them is lexicographically smaller. This problem is usually solved sequentially and is essentially a finite state machine of $n + 3$ states each having two transitions. The result is an almost identical decomposition (up to Clifford operations) with the same T complexity.

More concretely, consider how C/C++ [std::strcmp](https://en.cppreference.com/w/cpp/string/byte/strcmp) compares two sequences $A$ and $B$ of equal length $n$. The function scans the sequences from left to right until the first index $i^*$ where they differ and returns $A_{i^*} < B_{i^*}$.

Implicitly this algorithm has $n + 3$ states $\{e_0, \ldots, e_n\} \cup \{L, R\}$ where being in the $e_k$ state means the prefixes of length $k$ are equal. with transitions being the states governed by:

\begin{equation}
\begin{split}
    e_k \rightarrow e_{k+1} \textit{ if } u_k = v_k \\
    e_k \rightarrow L \textit{ if } u_k < v_k \\
    e_k \rightarrow R \textit{ if } u_k > v_k \\
\end{split}
\end{equation}

When the result of comparison between individual indices becomes probabilistic these states form a Markov decision process with three terminal states $\{L, e_n, R\}$. This gives us inspiration for a new implementation.

### Algorithm
We start by allocating $n+1$ qubits representing the $e_0, \ldots, e_n$ states and then scan the qubit register and number from left to right.

if the current bit is zero then we only need to compute the $e_k \rightarrow e_{k+1}$ transition since the qubit can't be less than zero. otherwise, we need to compute the transition as well as the $e_k → L$ transition.

In [ ]:
less_than = linear_constant_comparison

As we did before we construct the less than gate for checking if a 3 registers are less than 5. First, we print the decomposition of the gate followed by the result of running the gate on each of 8 possiblities individually and finally the result of running the gate on the uniform superposition of all 8 possibilities.

In [ ]:
less_than_circuit = less_than()
less_than_circuit.instructions

In [ ]:
show_counts(less_than_circuit, source_t=4 * 3)

In [ ]:
check_basis(less_than_circuit, 3, 1, lambda v: int(v < 5))

In [ ]:
check_uniform(less_than_circuit, 3, 1, lambda v: int(v < 5))

And as before, notice that the ancilla qubits are always clean at the end of execution and that the input qubits are not affected.

## Improving the constant

The implementation above has a T complexity of exactly $4n$ since there are exactly $n$ And gates each uses $4$ Ts. Note however that the first of them is not needed since one of its inputs is in the $\ket{1}$ state so it collapses to either identity or `cirq.X` depending on the most significant bit of $B$. This gives a T complexity of $4(n-1) = 4n - 4$.

## Citation

```
@article{Noureldin_2023,
	doi = {10.5281/zenodo.8384491},
	url = {https://doi.org/10.5281/zenodo.8384491},  
	year = 2023,
	author = {Noureldin Yosri},  
	title = {T Complexity Of Comparison Gates},
}
```